# Day 11 — Snowpark ML Fundamentals

Snowflake Notebook attached to `LEARN_WH`. Python, SQL (via `session.sql`), and Markdown
all run on warehouse compute. Do not `to_pandas()` the training set.


## 1. Session can read `RETAIL_LAKEHOUSE`


In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col

session = get_active_session()
session.sql("SELECT CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE()").show()
session.table("RETAIL_LAKEHOUSE.GOLD.ORDER_FACTS").limit(5).show()


## 2. Forecast with zero model building

`GOLD.ORDERS_OVER_TIME` has only 2 days — too short for `SNOWFLAKE.ML.FORECAST`.
`GOLD.DAILY_REVENUE` is 2,406 daily points from TPC-H SF1 (1992-01-01 .. 1998-08-02).

Trained model: `GOLD.DAILY_REVENUE_FORECAST`. 14-day output starts 1998-08-03 at ~94.16M.


In [ ]:
session.sql("""
SELECT MIN(TS) AS first_day, MAX(TS) AS last_day, COUNT(*) AS n_days
FROM RETAIL_LAKEHOUSE.GOLD.DAILY_REVENUE
""").show()

session.sql("""
SELECT TS, FORECAST, LOWER_BOUND, UPPER_BOUND
FROM TABLE(RETAIL_LAKEHOUSE.GOLD.DAILY_REVENUE_FORECAST!FORECAST(
    FORECASTING_PERIODS => 14
))
ORDER BY TS
""").show()


## 3. Preprocess with Snowpark ML

Load `GOLD.ML_ORDERS` (3,000-row Gold training table) as a Snowpark DataFrame.
Fit `StandardScaler` and `OneHotEncoder` in Snowflake — no local pandas copy.


In [ ]:
from snowflake.ml.modeling.preprocessing import StandardScaler, OneHotEncoder

df = session.table("RETAIL_LAKEHOUSE.GOLD.ML_ORDERS")
print("rows", df.count(), "cols", df.columns)

scaler = StandardScaler(
    input_cols=["O_TOTALPRICE", "O_SHIPPRIORITY"],
    output_cols=["O_TOTALPRICE_SCALED", "O_SHIPPRIORITY_SCALED"],
    passthrough_cols=["O_ORDERPRIORITY", "IS_FULFILLED"],
    drop_input_cols=True,
)
scaled = scaler.fit(df).transform(df)

ohe = OneHotEncoder(
    input_cols=["O_ORDERPRIORITY"],
    output_cols=["O_ORDERPRIORITY_OHE"],
    handle_unknown="ignore",
    passthrough_cols=["O_TOTALPRICE_SCALED", "O_SHIPPRIORITY_SCALED", "IS_FULFILLED"],
    drop_input_cols=True,
)
encoded = ohe.fit(scaled).transform(scaled)
encoded.select(
    col("O_TOTALPRICE_SCALED"),
    col("IS_FULFILLED"),
).limit(5).show()
print("encoded cols", encoded.columns)


## 4–6. Train RandomForestClassifier and score accuracy

Same `.fit()` / `.predict()` interface as scikit-learn. Training runs as a
warehouse stored procedure (add `pandas` / `scikit-learn` in the Notebook
packages list). Features: scaled price + ship priority + one-hot order priority.
Label: `IS_FULFILLED`.


In [ ]:
from snowflake.ml.modeling.ensemble import RandomForestClassifier
from snowflake.ml.modeling.metrics import accuracy_score

feature_cols = [c for c in encoded.columns if c not in ("IS_FULFILLED", "O_ORDERKEY", "O_ORDERSTATUS")]
train_df, test_df = encoded.select(*feature_cols, "IS_FULFILLED").random_split([0.8, 0.2], seed=42)

model = RandomForestClassifier(
    input_cols=feature_cols,
    label_cols=["IS_FULFILLED"],
    output_cols=["PREDICTED"],
    n_estimators=20,
    max_depth=6,
    random_state=42,
)
model.fit(train_df)
pred = model.predict(test_df)
acc = accuracy_score(df=pred, y_true_col_names="IS_FULFILLED", y_pred_col_names="PREDICTED")
print("accuracy_score", acc)
pred.select("IS_FULFILLED", "PREDICTED").limit(8).show()


## Why in-platform training

See `docs/day11_training_notes.md`. Short version: `to_pandas()` + local sklearn
moves Gold data off the platform (scale + governance). Snowpark ML and
`SNOWFLAKE.ML.FORECAST` keep compute, RBAC, and masking next to the tables.
